<a href="https://colab.research.google.com/github/RAJAMURUGAN-VS/genai-learning-journey/blob/main/02-groq-api/03_function_calling_weather_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Groq package

In [9]:
!pip install groq

Import necessary classes and functions

In [10]:
from google.colab import userdata
from groq import Groq
import json
import requests

Create client object using Groq class

In [11]:
client=Groq(
  api_key=userdata.get('GROQ_API_KEY')
)
print(client)

Function to get Weather Information

In [12]:
def get_weather(location):
 api_key = userdata.get('WEATHER_API_KEY')
 url = f"http://api.openweathermap.org/data/2.5/weather?q={location}&units=metric&appid={api_key}"
 response = requests.get(url)
 data = response.json()
 print(data)

 if data["cod"] == 200:
   return {
     "location": location,
     "temperature": data["main"]["temp"],
     "description": data["weather"][0]["description"]
   }
 else:
   return {"Oops! Something went wrong."}

Tool Definition for get_weather function

In [13]:
tools = [
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "Get current weather for a city",
      "parameters": {
        "type": "object",
        "properties": {
          "location": {
            "type": "string",
            "description": "City name like Mumbai, London"
            }
            },
      "required": ["location"]
           }
       }
   }
]

Sending the Tool Definition to the Model

In [14]:
llm_messages = [
  {
    "role": "system",
    "content": "You are a weather assistant. Use get_weather function when asked about weather."
  },
  {
    "role": "user",
    "content": "What's the weather in Mumbai?"
  }
]

response = client.chat.completions.create(
  model="llama-3.3-70b-versatile",
  messages=llm_messages,
  tools=tools,
  tool_choice="auto"
)

print(response.model_dump_json(indent=2))

{
  "id": "chatcmpl-2f047047-5805-43c7-a374-e940a21d9081",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": null,
        "role": "assistant",
        "annotations": null,
        "executed_tools": null,
        "function_call": null,
        "reasoning": null,
        "tool_calls": [
          {
            "id": "93m9k0gw8",
            "function": {
              "arguments": "{\"location\":\"Mumbai\"}",
              "name": "get_weather"
            },
            "type": "function"
          }
        ]
      }
    }
  ],
  "created": 1779771460,
  "model": "llama-3.3-70b-versatile",
  "object": "chat.completion",
  "mcp_list_tools": null,
  "service_tier": "on_demand",
  "system_fingerprint": "fp_0761e44d7b",
  "usage": {
    "completion_tokens": 15,
    "prompt_tokens": 243,
    "total_tokens": 258,
    "completion_time": 0.053628695,
    "completion_tokens_details": null,
    "prompt_time"

In [15]:
response_message = response.choices[0].message
print(response_message.model_dump_json(indent=2))

{
  "content": null,
  "role": "assistant",
  "annotations": null,
  "executed_tools": null,
  "function_call": null,
  "reasoning": null,
  "tool_calls": [
    {
      "id": "93m9k0gw8",
      "function": {
        "arguments": "{\"location\":\"Mumbai\"}",
        "name": "get_weather"
      },
      "type": "function"
    }
  ]
}
